# CIFAR-10

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time
import pandas as pd
from torch.optim.lr_scheduler import CosineAnnealingLR
from thop import profile  # For calculating FLOPs
import math

GLOBAL_TRAIN_BATCH_INDICES = None

# GPU Configuration
def configure_gpu():
    """
    Configure GPU settings for PyTorch
    """
    # Check for GPU availability
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Print number of available GPUs
    if torch.cuda.is_available():
        print(f"Num GPUs Available: {torch.cuda.device_count()}")

        # Configure GPU memory management
        torch.cuda.empty_cache()
        print("GPU memory cleared")

    return device

def load_images(batch_size=128, val_size=5000):
    """
    Loads CIFAR-10 with train/val/test split:
    - Training: 45,000 samples
    - Validation: 5,000 samples
    - Test: 10,000 samples

    Uses consistent batch composition with efficient sampling for training.
    """
    # Define transformations
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ])

    transform_val_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ])

    # Load full training dataset (50,000 samples)
    # Use CIFAR100 instead of CIFAR10 to run it with CIFAR100. Change the normalization values and class numbers accordingly.
    full_train_dataset = datasets.CIFAR10(
        root='./data',
        train=True,
        download=True,
        transform=transform_train
    )

    # Load validation dataset with test transforms (no augmentation)
    full_train_dataset_for_val = datasets.CIFAR10(
        root='./data',
        train=True,
        download=True,
        transform=transform_val_test
    )

    # Load test dataset
    test_dataset = datasets.CIFAR10(
        root='./data',
        train=False,
        download=True,
        transform=transform_val_test
    )

    class_names = ['plane', 'car', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

    # --- Split training data into train (45k) and validation (5k) ---
    num_total_train = len(full_train_dataset)  # 50,000
    indices = np.arange(num_total_train)
    np.random.shuffle(indices)

    # Split indices
    val_indices = indices[:val_size]  # First 5,000 for validation
    train_indices = indices[val_size:]  # Remaining 45,000 for training

    print(f"\nData Split:")
    print(f"  Training samples: {len(train_indices)}")
    print(f"  Validation samples: {len(val_indices)}")
    print(f"  Test samples: {len(test_dataset)}")

    # Create validation subset (using non-augmented transforms)
    val_dataset = Subset(full_train_dataset_for_val, val_indices)

    # --- Efficient Consistent Batch Handling for Training ---
    # Shuffle training indices and group into batches
    np.random.shuffle(train_indices)

    # Group into batches
    batch_indices = [
        train_indices[i:i + batch_size].tolist()
        for i in range(0, len(train_indices), batch_size)
    ]

    # Save batch indices for later access in training loop
    global GLOBAL_TRAIN_BATCH_INDICES
    GLOBAL_TRAIN_BATCH_INDICES = batch_indices  # Make accessible for pruning logic

    print(f"\nCreated consistent batch setup with {len(batch_indices)} training batches")
    print(f"Each batch contains the same data samples across all epochs")
    print(f"Intra-batch shuffling is enabled to prevent memorization")

    # Validation DataLoader
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Test DataLoader
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    return full_train_dataset, val_loader, test_loader, class_names


def build_train_loader_for_epoch(full_train_dataset, active_batch_indices, batch_size, shuffle_within_batch=True):

    global GLOBAL_TRAIN_BATCH_INDICES

    # Collect (logical_batch_idx, sample_indices) for active batches only
    active_batches = []
    for logical_idx in active_batch_indices:
        sample_indices = GLOBAL_TRAIN_BATCH_INDICES[logical_idx].copy()
        if shuffle_within_batch:
            np.random.shuffle(sample_indices)
        active_batches.append((logical_idx, sample_indices))

    # Flatten sample indices in order, and record which logical batch each sample belongs to
    flat_indices = []
    logical_index_order = []  # One entry per batch, in iteration order
    for logical_idx, sample_indices in active_batches:
        flat_indices.extend(sample_indices)
        logical_index_order.append(logical_idx)

    # Custom sampler that yields the flat indices in exact order
    class OrderedIndexSampler:
        def __init__(self, indices):
            self.indices = indices

        def __iter__(self):
            return iter(self.indices)

        def __len__(self):
            return len(self.indices)

    sampler = OrderedIndexSampler(flat_indices)

    train_loader = DataLoader(
        full_train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=4,
        shuffle=False,
        pin_memory=True,
        drop_last=False
    )

    return train_loader, logical_index_order


# Updated ResNet model definitions from paste-2.txt
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion *
                               planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        # List to track layer names for activation tracking
        self.layer_names = ['layer1', 'layer2', 'layer3', 'layer4']

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.dropout = nn.Dropout(0.2)  # Adding dropout for regularization
        self.linear = nn.Linear(512*block.expansion, num_classes)

        # List of layers to track activations
        self.conv_layers = [self.layer1, self.layer2, self.layer3, self.layer4]

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_activations=False):
        # Initialize list to store activations if requested
        activations = []

        # Initial convolution
        out = F.relu(self.bn1(self.conv1(x)))

        # ResNet block 1
        out = self.layer1(out)
        if return_activations:
            activations.append(out.detach())

        # ResNet block 2
        out = self.layer2(out)
        if return_activations:
            activations.append(out.detach())

        # ResNet block 3
        out = self.layer3(out)
        if return_activations:
            activations.append(out.detach())

        # ResNet block 4
        out = self.layer4(out)
        if return_activations:
            activations.append(out.detach())

        # Global average pooling and final classifier
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.dropout(out)  # Apply dropout before final layer
        out = self.linear(out)

        if return_activations:
            return out, activations
        return out

    def create_emb(self, x):
        """
        Extract embeddings from the model
        """
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return out

def ResNet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

def ResNet50(num_classes=10):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes)

def calculate_model_parameters(model):
    """
    Calculate the total number of parameters in a model
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\nModel Parameters:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    return total_params, trainable_params

def calculate_flops(model, input_size=(3, 32, 32)):
    """
    Calculate FLOPs for the model
    """
    # Create a dummy input based on the specified input size
    device = next(model.parameters()).device
    dummy_input = torch.randn(1, *input_size).to(device)

    # Calculate FLOPs
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)

    # Print results
    print(f"\nModel Computational Requirements:")
    print(f"FLOPs: {flops:,} ({flops/1e9:.2f} GFLOPs)")

    return flops


def train_epoch(model, device, train_loader, optimizer, epoch, logical_index_order, scheduler=None, threshold=0.0001):

    model.train()
    train_loss = 0
    correct = 0
    total = 0

    # Initialize storage for standard deviations per layer
    std_devs = {layer_name: [] for layer_name in model.layer_names}
    batch_indices_list = []

    # Track throughput
    start_time = time.time()
    total_images_processed = 0
    batch_processing_times = []

    num_active_batches = len(logical_index_order)
    print(f"Using {num_active_batches} batches this epoch")

    # Iterate over the loader; each batch aligns with logical_index_order
    for loader_batch_idx, (data, target) in enumerate(train_loader):
        # The loader_batch_idx-th batch from the loader corresponds to
        # logical batch logical_index_order[loader_batch_idx]
        logical_batch_idx = logical_index_order[loader_batch_idx]

        batch_start_time = time.time()
        batch_indices_list.append(logical_batch_idx)

        # Move to device
        data, target = data.to(device), target.to(device)

        # Optional: Mixup augmentation
        if np.random.random() > 0.5:
            lam = np.random.beta(0.2, 0.2)
            rand_idx = torch.randperm(data.size(0)).to(device)
            mixed_data = lam * data + (1 - lam) * data[rand_idx]
            target_a, target_b = target, target[rand_idx]

            optimizer.zero_grad()
            output, activations = model(mixed_data, return_activations=True)
            loss = lam * F.cross_entropy(output, target_a) + (1 - lam) * F.cross_entropy(output, target_b)
        else:
            optimizer.zero_grad()
            output, activations = model(data, return_activations=True)
            loss = F.cross_entropy(output, target)

        # Compute standard deviation of activations for each layer
        for layer_idx, act in enumerate(activations):
            std_dev = torch.std(act).item()
            std_devs[model.layer_names[layer_idx]].append(std_dev)

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Accumulate loss and accuracy
        train_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

        # Throughput tracking
        batch_end_time = time.time()
        batch_time = batch_end_time - batch_start_time
        batch_processing_times.append(batch_time)
        total_images_processed += data.size(0)

        # Print progress
        if (loader_batch_idx + 1) % 100 == 0:
            print(f'Train Epoch: {epoch} [{total_images_processed}/{num_active_batches * data.size(0)} '
                  f'({100. * (loader_batch_idx + 1) / num_active_batches:.0f}%)]\t'
                  f'Loss: {loss.item():.6f}\tAccuracy: {100. * correct / total:.2f}%')

    # End timing
    end_time = time.time()
    training_time = end_time - start_time

    # Build DataFrame of standard deviations
    std_df = pd.DataFrame(std_devs)
    std_df['batch_idx'] = batch_indices_list
    std_df = std_df.set_index('batch_idx')
    std_df['mean_std'] = std_df.mean(axis=1)

    # Throughput metrics
    avg_batch_time = np.mean(batch_processing_times) if batch_processing_times else 0
    images_per_second = total_images_processed / training_time if training_time > 0 else 0

    # Average loss and accuracy
    epoch_loss = train_loss / len(batch_indices_list) if batch_indices_list else 0
    epoch_acc = 100. * correct / total if total > 0 else 0

    # Step scheduler
    if scheduler is not None:
        scheduler.step()

    print(f"\nEpoch {epoch} completed in {training_time:.2f} seconds")
    print(f"Training Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")
    print(f"Throughput: {images_per_second:.2f} images/second, Avg batch time: {avg_batch_time:.4f} s")
    print(f"Current threshold: {threshold:.8f}")
    print(f"Batch composition: CONSISTENT across epochs (same samples per batch)")
    print(f"Intra-batch order: SHUFFLED to prevent memorization")

    throughput_metrics = {
        'epoch_time': training_time,
        'images_per_second': images_per_second,
        'avg_batch_time': avg_batch_time,
        'total_batches': len(batch_indices_list)
    }

    return epoch_loss, epoch_acc, std_df, batch_indices_list, throughput_metrics


def validate(model, device, data_loader, dataset_name="Validation"):
    """
    Validates/tests the model on a given dataset
    """
    model.eval()
    val_loss = 0
    correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in data_loader:
            # Move data to device
            data, target = data.to(device), target.to(device)

            # Forward pass
            output = model(data)

            # Compute loss
            val_loss += F.cross_entropy(output, target).item()

            # Calculate accuracy
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()

            # Store predictions and targets for confusion matrix
            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    # Calculate average loss and accuracy
    val_loss /= len(data_loader)
    val_acc = 100. * correct / len(data_loader.dataset)

    print(f"{dataset_name} Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")

    return val_loss, val_acc, all_targets, all_preds

def plot_training_metrics(epoch_times, train_losses, train_accs, val_losses, val_accs):
    """
    Plots metrics related to training time and performance
    """
    plt.figure(figsize=(18, 12))

    # Plot 1: Training Time per Epoch
    plt.subplot(2, 2, 1)
    plt.plot(range(1, len(epoch_times) + 1), epoch_times, marker='o')
    plt.title('Training Time per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Time (seconds)')
    plt.grid(True)

    # Plot 2: Accuracy
    plt.subplot(2, 2, 3)
    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(val_accs, label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Plot 3: Loss
    plt.subplot(2, 2, 4)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('training_metrics_improved_woes.pdf')
    plt.show()

def plot_confusion_matrix(all_targets, all_preds, class_names, title='Confusion Matrix'):
    """
    Plots the confusion matrix
    """
    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title(title)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_improved_woes.pdf')
    plt.show()

def calculate_exponential_threshold(delta_start, delta_end, T, t):
    """
    Calculate threshold using exponential function: δ(t) = δ_initial * e^(alpha * t)

    Args:
        delta_start (float): Initial threshold value
        delta_end (float): Final threshold value
        T (int): Total number of epochs
        t (int): Current epoch number

    Returns:
        float: Threshold value for the current epoch
    """

    # Compute alpha
    alpha = math.log(delta_end / delta_start) / T
    threshold = delta_start * math.exp(alpha * t)
    return threshold



def plot_batch_prune_analysis(threshold_history, remaining_batches_history,
                            dropped_per_epoch_history, pruned_percentage_history):
    """
    Create comprehensive visualizations for batch pruning analysis with exponential threshold
    """
    plt.figure(figsize=(20, 12))

    # Plot 1: Threshold evolution over epochs
    plt.subplot(3, 2, 1)
    plt.plot(range(1, len(threshold_history) + 1), threshold_history, marker='o', color='red')
    plt.title('Exponential Threshold Evolution Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Threshold Value')
    plt.yscale('log')  # Use log scale for better visualization
    plt.grid(True, alpha=0.3)

    # Plot 2: Remaining batches over epochs
    plt.subplot(3, 2, 2)
    plt.plot(range(1, len(remaining_batches_history) + 1), remaining_batches_history, marker='o', color='blue')
    plt.title('Remaining Batches per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Batches')
    plt.grid(True, alpha=0.3)

    # Plot 3: Batches dropped per epoch
    plt.subplot(3, 2, 3)
    plt.bar(range(1, len(dropped_per_epoch_history) + 1), dropped_per_epoch_history, color='orange', alpha=0.7)
    plt.title('Batches Dropped per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Batches Dropped')
    plt.grid(True, alpha=0.3)

    # Plot 4: Cumulative pruned percentage over epochs
    plt.subplot(3, 2, 4)
    plt.plot(range(1, len(pruned_percentage_history) + 1), pruned_percentage_history,
             marker='d', color='green', linewidth=2)
    plt.title('Cumulative Pruned Percentage Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Pruned Percentage (%)')
    plt.grid(True, alpha=0.3)

    # Plot 5: Threshold vs Remaining Batches (correlation analysis)
    plt.subplot(3, 2, 5)
    plt.scatter(threshold_history, remaining_batches_history, alpha=0.6, color='coral')
    plt.title('Threshold vs Remaining Batches')
    plt.xlabel('Threshold Value')
    plt.ylabel('Remaining Batches')
    plt.xscale('log')
    plt.grid(True, alpha=0.3)

    # Plot 6: Exponential threshold growth pattern
    plt.subplot(3, 2, 6)
    epochs = range(1, len(threshold_history) + 1)
    plt.semilogy(epochs, threshold_history, marker='s', color='purple', linewidth=2)
    plt.title('Exponential Threshold Growth Pattern')
    plt.xlabel('Epoch')
    plt.ylabel('Threshold Value (log scale)')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('batch_prune_exponential_threshold_analysis.pdf', dpi=300, bbox_inches='tight')
    plt.show()

def run_cifar10_classification():
    # Configure GPU
    device = configure_gpu()

    total_epochs = 200
    batch_size = 128
    learning_rate = 0.05
    weight_decay = 5e-4
    momentum = 0.9


    # Exponential threshold parameters
    delta_start = 0.000001   # starting δ
    delta_end = 0.00005    # ending δ


    min_batch_percentage = 0  # Minimum percentage of batches to keep before stopping

    # Load data with train/val/test split (45k/5k/10k)
    full_train_dataset, val_loader, test_loader, class_names = load_images(batch_size, val_size=5000)

    # Calculate max batches per epoch
    global GLOBAL_TRAIN_BATCH_INDICES
    total_original_batches = len(GLOBAL_TRAIN_BATCH_INDICES)
    max_epochs = total_epochs  # for DUI calculation

    print(f"Total original training batches per epoch: {total_original_batches}")
    print(f"Maximum number of epochs: {max_epochs}")

    # Create improved ResNet18 model
    model = ResNet18(num_classes=10).to(device)

    # Print model summary
    print(model)

    # Calculate model parameters
    total_params, trainable_params = calculate_model_parameters(model)

    # Calculate FLOPs
    try:
        flops = calculate_flops(model)
    except ImportError:
        print("thop package not installed. Skipping FLOPs calculation.")
        flops = None

    # Define improved optimizer and scheduler with OneCycleLR
    optimizer = optim.SGD(model.parameters(),
                         lr=learning_rate,
                         momentum=momentum,
                         weight_decay=weight_decay,
                         nesterov=True)  # Enable Nesterov momentum

    # CosineAnnealingLR schedule for better convergence
    scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=1e-5)

    # Training and validation history
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []

    # Metrics to track
    epoch_times = []
    batch_counts = []

    # Initialize tracking for batch dropping with exponential threshold
    prev_epoch_std_df = None
    batch_indices_to_use = list(range(total_original_batches))  # Start with all batches
    total_dropped_batches = 0
    total_remaining_batches = []  # Track remaining batches for each epoch
    total_batches_dropped_per_epoch = []  # Track batches dropped per epoch

    # New tracking for exponential threshold
    threshold_history = []  # Track threshold values over epochs
    pruned_percentage_history = []  # Track cumulative pruned percentage

    # Visualizations for batch dropping analysis
    batch_std_history = {}  # Dictionary to store std values by batch across epochs

    # Track total training time
    total_training_start = time.time()


    best_val_acc = 0
    best_model_state = None

    # Train and evaluate
    for epoch in range(1, total_epochs + 1):
        print(f"\n{'='*50}")
        print(f"EPOCH {epoch}/{total_epochs}")
        print(f"{'='*50}")

        # Calculate threshold using exponential function
        current_threshold = calculate_exponential_threshold(delta_start, delta_end, total_epochs, epoch)

        # Record threshold history
        threshold_history.append(current_threshold)

        print(f"Current threshold (δ({epoch})): {current_threshold:.8f}")


        train_loader, logical_index_order = build_train_loader_for_epoch(
            full_train_dataset, batch_indices_to_use, batch_size, shuffle_within_batch=True
        )

        # Train for one epoch
        train_loss, train_acc, current_std_df, used_batch_indices, throughput_metrics = train_epoch(
            model, device, train_loader, optimizer, epoch,
            logical_index_order=logical_index_order,
            scheduler=scheduler,
            threshold=current_threshold
        )

        # Record metrics
        epoch_times.append(throughput_metrics['epoch_time'])
        batch_counts.append(throughput_metrics['total_batches'])
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # Track remaining batches for this epoch
        current_remaining_batches = len(used_batch_indices) if used_batch_indices is not None else total_original_batches
        total_remaining_batches.append(current_remaining_batches)

        # Calculate and record pruned percentage
        current_pruned_percentage = (1 - current_remaining_batches / total_original_batches) * 100
        pruned_percentage_history.append(current_pruned_percentage)

        # Evaluate on VALIDATION set (not test set)
        val_loss, val_acc, _, _ = validate(model, device, val_loader, dataset_name="Validation")
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        # Save best model so far (based on validation accuracy)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {key: value.cpu().clone() for key, value in model.state_dict().items()}
            print(f"New best validation accuracy: {best_val_acc:.2f}%")
            # Save model to disk
            torch.save(model.state_dict(), 'best_resnet18_exponential_threshold.pth')

        # Track batches dropped for this epoch
        batches_dropped_this_epoch = 0

        # After the first epoch, we start the batch dropping mechanism
        if epoch > 1 and prev_epoch_std_df is not None:
            # Find matching batch indices between previous and current epoch
            common_indices = set(prev_epoch_std_df.index).intersection(set(current_std_df.index))

            # Initialize list to track which batches to drop
            batches_to_drop = []

            # Compare mean standard deviations for each batch using exponential threshold
            for idx in common_indices:
                prev_mean_std = prev_epoch_std_df.loc[idx, 'mean_std']
                curr_mean_std = current_std_df.loc[idx, 'mean_std']

                # Update the batch std history for visualization
                if idx not in batch_std_history:
                    batch_std_history[idx] = []
                batch_std_history[idx].append(curr_mean_std)

                # If the absolute difference is below the exponential threshold, mark this batch for dropping
                if abs(prev_mean_std - curr_mean_std) <= current_threshold:
                    batches_to_drop.append(idx)

            # Print information about dropped batches
            if batches_to_drop:
                print(f"\nDropping {len(batches_to_drop)} batches for the next epoch:")
                print(f"Batch indices: {batches_to_drop[:10]}{'...' if len(batches_to_drop) > 10 else ''}")
                total_dropped_batches += len(batches_to_drop)
                batches_dropped_this_epoch = len(batches_to_drop)
            else:
                print("\nNo batches to drop for the next epoch")

            # Update batch indices for the next epoch (exclude the ones to drop)
            batch_indices_to_use = [i for i in batch_indices_to_use if i not in set(batches_to_drop)]

        # Save current dataframe for comparison with next epoch
        prev_epoch_std_df = current_std_df

        # Record batches dropped this epoch
        total_batches_dropped_per_epoch.append(batches_dropped_this_epoch)

        # Calculate percent of original batches remaining
        percent_remaining = len(batch_indices_to_use) / total_original_batches * 100
        print(f"\nBatches remaining: {len(batch_indices_to_use)}/{total_original_batches} ({percent_remaining:.2f}%)")
        print(f"Total batches dropped so far: {total_dropped_batches}/{total_original_batches} ({total_dropped_batches/total_original_batches*100:.2f}%)")

        # Plot std distribution of remaining batches vs dropped batches for better analysis
        if epoch % 10 == 0:  # Only plot every 10 epochs to avoid clutter
            plot_batch_std_distribution(current_std_df, batch_indices_to_use, epoch)

        # Early stopping if we've dropped too many batches
        if len(batch_indices_to_use) <= total_original_batches * min_batch_percentage:
            print(f"\nStopping early as fewer than {min_batch_percentage*100}% of original batches remain")
            break

    # If we have a best model state, load it back
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model with validation accuracy: {best_val_acc:.2f}%")

    # Calculate total training time
    total_training_end = time.time()
    total_training_time = total_training_end - total_training_start
    average_epoch_time = sum(epoch_times) / len(epoch_times) if epoch_times else 0

    # Calculate Data Utilization Index (DUI)
    total_epochs_completed = len(epoch_times)
    sum_remaining_batches = sum(total_remaining_batches)
    max_possible_batches = max_epochs * total_original_batches
    data_utilization_index = sum_remaining_batches / max_possible_batches
    data_savings_index = 1 - data_utilization_index
    # Plot batch pruning analysis with exponential threshold
    plot_batch_prune_analysis(threshold_history, total_remaining_batches,
                            total_batches_dropped_per_epoch, pruned_percentage_history)

    # Plot original batch dropping analysis for comparison
    plot_batch_dropping_analysis(total_remaining_batches, total_batches_dropped_per_epoch, epoch_times, batch_std_history)

    # Final stats on batch dropping
    percent_dropped = total_dropped_batches / total_original_batches * 100
    percent_remaining = 100 - percent_dropped
    total_remaining_batch = total_original_batches - total_dropped_batches
    print(f"\nData Utilization Index (DUI): {data_utilization_index:.4f}")
    print(f"\nData Savings Index (DSI): {data_savings_index:.4f}")
    print(f"Sum of remaining batches across all epochs: {sum_remaining_batches}")
    print(f"Maximum possible batches (max_epochs * total_batches): {max_possible_batches}")

    print(f"\nFinal Statistics:")
    print(f"Total batches dropped: {total_dropped_batches}/{total_original_batches} ({percent_dropped:.2f}%)")
    print(f"Total batches remaining: {total_remaining_batch}/{total_original_batches} ({percent_remaining:.2f}%)")
    print(f"Final threshold: {threshold_history[-1]:.8f}")

    # Output model statistics and results
    print(f"\nTraining Time Statistics:")
    print(f"Total training time: {total_training_time:.2f} seconds ({total_training_time/60:.2f} minutes)")
    print(f"Average epoch time: {average_epoch_time:.2f} seconds")
    print(f"Number of epochs completed: {len(epoch_times)}")

    # Print model statistics
    print(f"\nModel Statistics:")
    print(f"Total parameters: {total_params:,}")
    if flops:
        print(f"FLOPs: {flops:,} ({flops/1e9:.2f} GFLOPs)")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    # Perform final validation on the VALIDATION set
    print(f"\n{'='*50}")
    print("FINAL VALIDATION SET EVALUATION")
    print(f"{'='*50}")
    final_val_loss, final_val_acc, final_val_targets, final_val_preds = validate(
        model, device, val_loader, dataset_name="Final Validation"
    )

    # Perform final evaluation on the TEST set
    print(f"\n{'='*50}")
    print("FINAL TEST SET EVALUATION")
    print(f"{'='*50}")
    final_test_loss, final_test_acc, final_test_targets, final_test_preds = validate(
        model, device, test_loader, dataset_name="Final Test"
    )

    print(f"\nFinal Results Summary:")
    print(f"  Validation Loss: {final_val_loss:.4f}, Accuracy: {final_val_acc:.2f}%")
    print(f"  Test Loss: {final_test_loss:.4f}, Accuracy: {final_test_acc:.2f}%")

    # Plot confusion matrix for TEST set evaluation
    plot_confusion_matrix(final_test_targets, final_test_preds, class_names, title='Test Set Confusion Matrix')

    # Plot training metrics
    plot_training_metrics(epoch_times, train_losses, train_accs, val_losses, val_accs)

    # Create and save final summary DataFrame with exponential threshold info
    summary_data = {
        'total_training_time': [total_training_time],
        'average_epoch_time': [average_epoch_time],
        'epochs_completed': [total_epochs_completed],
        'best_validation_accuracy': [best_val_acc],
        'final_validation_accuracy': [final_val_acc],
        'final_test_accuracy': [final_test_acc],
        'total_parameters': [total_params],
        'data_utilization_index': [data_utilization_index],
        'data_savings_index': [data_savings_index],
        'total_batches_dropped': [total_dropped_batches],
        'total_remaining_batches': [total_remaining_batch],
        'batch_drop_percentage': [percent_dropped],
        'final_threshold': [threshold_history[-1] if threshold_history else delta_start],
        'learning_rate': [learning_rate],
        'weight_decay': [weight_decay],
        'train_samples': [45000],
        'val_samples': [5000],
        'test_samples': [10000]
    }

    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv('exponential_threshold_batch_dropping_summary.csv', index=False)
    print("\nSummary saved to exponential_threshold_batch_dropping_summary.csv")

    # Save threshold history for further analysis
    threshold_df = pd.DataFrame({
        'epoch': range(1, len(threshold_history) + 1),
        'threshold': threshold_history,
        'remaining_batches': total_remaining_batches,
        'batches_dropped': total_batches_dropped_per_epoch,
        'pruned_percentage': pruned_percentage_history
    })
    threshold_df.to_csv('exponential_threshold_history.csv', index=False)
    print("Threshold history saved to exponential_threshold_history.csv")

    return model, summary_df

def plot_batch_std_distribution(std_df, remaining_batch_indices, epoch):
    """
    Plots the distribution of standard deviations for remaining vs dropped batches
    """
    plt.figure(figsize=(10, 6))

    # Separate std values for remaining and dropped batches
    remaining_stds = std_df.loc[std_df.index.isin(remaining_batch_indices), 'mean_std']
    dropped_stds = std_df.loc[~std_df.index.isin(remaining_batch_indices), 'mean_std']

    # Plot histograms
    plt.hist(remaining_stds, bins=30, alpha=0.5, label='Remaining Batches')
    if len(dropped_stds) > 0:
        plt.hist(dropped_stds, bins=30, alpha=0.5, label='Dropped Batches')

    plt.title(f'Batch Std Distribution - Epoch {epoch}')
    plt.xlabel('Mean Standard Deviation')
    plt.ylabel('Number of Batches')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f'batch_std_distribution_epoch_{epoch}_woes.pdf')
    plt.close()

def plot_batch_dropping_analysis(remaining_batches, dropped_per_epoch, epoch_times, batch_std_history):
    """
    Create comprehensive visualizations for batch dropping analysis
    """
    plt.figure(figsize=(18, 12))

    # Plot 1: Remaining batches over epochs
    plt.subplot(2, 2, 1)
    plt.plot(range(1, len(remaining_batches) + 1), remaining_batches, marker='o')
    plt.title('Remaining Batches per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Batches')
    plt.grid(True)

    # Plot 2: Batches dropped per epoch
    plt.subplot(2, 2, 2)
    plt.bar(range(1, len(dropped_per_epoch) + 1), dropped_per_epoch)
    plt.title('Batches Dropped per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Batches Dropped')
    plt.grid(True)

    # Plot 3: Training time vs remaining batches
    plt.subplot(2, 2, 3)
    plt.scatter(remaining_batches, epoch_times)
    plt.title('Training Time vs Remaining Batches')
    plt.xlabel('Number of Batches')
    plt.ylabel('Epoch Time (seconds)')
    plt.grid(True)

    # Plot 4: Standard deviation trend for a sample of batches
    plt.subplot(2, 2, 4)
    # Sample up to 10 batches for visualization
    sample_batches = list(batch_std_history.keys())[:10]
    for batch_idx in sample_batches:
        std_values = batch_std_history[batch_idx]
        plt.plot(range(1, len(std_values) + 1), std_values, label=f'Batch {batch_idx}')

    plt.title('Std Deviation Trend for Sample Batches')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Std Deviation_woes')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('batch_dropping_analysis.pdf')
    plt.close()

if __name__ == "__main__":
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Run the entire classification process
    model, summary_df = run_cifar10_classification()

# SVHN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time
import pandas as pd
from torch.optim.lr_scheduler import CosineAnnealingLR
from thop import profile  # For calculating FLOPs
import math

GLOBAL_TRAIN_BATCH_INDICES = None

# GPU Configuration
def configure_gpu():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    if torch.cuda.is_available():
        print(f"Num GPUs Available: {torch.cuda.device_count()}")
        torch.cuda.empty_cache()
        print("GPU memory cleared")

    return device

def load_images(batch_size=128, val_size=5000):
    """
    Loads SVHN with train/val/test split:
    - Training: 68,257 samples (73,257 - 5,000)
    - Validation: 5,000 samples
    - Test: 26,032 samples

    SVHN images are 32x32 RGB, with labels 0-9 (digit classes).
    Uses consistent batch composition with efficient sampling for training.
    """
    # SVHN normalization statistics
    svhn_mean = (0.4377, 0.4438, 0.4728)
    svhn_std = (0.1980, 0.2010, 0.1970)

    # Define transformations (SVHN typically uses lighter augmentation than CIFAR)
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(svhn_mean, svhn_std)
    ])

    transform_val_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(svhn_mean, svhn_std)
    ])

    # Load full training dataset (73,257 samples)
    full_train_dataset = datasets.SVHN(
        root='./data',
        split='train',
        download=True,
        transform=transform_train
    )

    # Load validation dataset with test transforms (no augmentation)
    full_train_dataset_for_val = datasets.SVHN(
        root='./data',
        split='train',
        download=True,
        transform=transform_val_test
    )

    # Load test dataset (26,032 samples)
    test_dataset = datasets.SVHN(
        root='./data',
        split='test',
        download=True,
        transform=transform_val_test
    )

    # SVHN has 10 digit classes
    class_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

    # --- Split training data into train and validation ---
    num_total_train = len(full_train_dataset)  # 73,257
    indices = np.arange(num_total_train)
    np.random.shuffle(indices)

    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    print(f"\nData Split (SVHN):")
    print(f"  Total training pool: {num_total_train}")
    print(f"  Training samples: {len(train_indices)}")
    print(f"  Validation samples: {len(val_indices)}")
    print(f"  Test samples: {len(test_dataset)}")
    print(f"  Number of classes: {len(class_names)}")

    # Create validation subset (using non-augmented transforms)
    val_dataset = Subset(full_train_dataset_for_val, val_indices)

    # --- Efficient Consistent Batch Handling for Training ---
    np.random.shuffle(train_indices)

    # Group into batches
    batch_indices = [
        train_indices[i:i + batch_size].tolist()
        for i in range(0, len(train_indices), batch_size)
    ]

    global GLOBAL_TRAIN_BATCH_INDICES
    GLOBAL_TRAIN_BATCH_INDICES = batch_indices

    print(f"\nCreated consistent batch setup with {len(batch_indices)} training batches")
    print(f"Each batch contains the same data samples across all epochs")
    print(f"Intra-batch shuffling is enabled to prevent memorization")

    # Validation DataLoader
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Test DataLoader
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    return full_train_dataset, val_loader, test_loader, class_names


def build_train_loader_for_epoch(full_train_dataset, active_batch_indices, batch_size, shuffle_within_batch=True):

    global GLOBAL_TRAIN_BATCH_INDICES

    active_batches = []
    for logical_idx in active_batch_indices:
        sample_indices = GLOBAL_TRAIN_BATCH_INDICES[logical_idx].copy()
        if shuffle_within_batch:
            np.random.shuffle(sample_indices)
        active_batches.append((logical_idx, sample_indices))

    flat_indices = []
    logical_index_order = []
    for logical_idx, sample_indices in active_batches:
        flat_indices.extend(sample_indices)
        logical_index_order.append(logical_idx)

    class OrderedIndexSampler:
        def __init__(self, indices):
            self.indices = indices

        def __iter__(self):
            return iter(self.indices)

        def __len__(self):
            return len(self.indices)

    sampler = OrderedIndexSampler(flat_indices)

    train_loader = DataLoader(
        full_train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=4,
        shuffle=False,
        pin_memory=True,
        drop_last=False
    )

    return train_loader, logical_index_order


# ResNet model definitions
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion *
                               planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.layer_names = ['layer1', 'layer2', 'layer3', 'layer4']

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.dropout = nn.Dropout(0.2)
        self.linear = nn.Linear(512*block.expansion, num_classes)

        self.conv_layers = [self.layer1, self.layer2, self.layer3, self.layer4]

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_activations=False):
        activations = []

        out = F.relu(self.bn1(self.conv1(x)))

        out = self.layer1(out)
        if return_activations:
            activations.append(out.detach())

        out = self.layer2(out)
        if return_activations:
            activations.append(out.detach())

        out = self.layer3(out)
        if return_activations:
            activations.append(out.detach())

        out = self.layer4(out)
        if return_activations:
            activations.append(out.detach())

        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.dropout(out)
        out = self.linear(out)

        if return_activations:
            return out, activations
        return out

    def create_emb(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return out

def ResNet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

def ResNet50(num_classes=10):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes)

def calculate_model_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\nModel Parameters:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    return total_params, trainable_params

def calculate_flops(model, input_size=(3, 32, 32)):
    device = next(model.parameters()).device
    dummy_input = torch.randn(1, *input_size).to(device)

    flops, params = profile(model, inputs=(dummy_input,), verbose=False)

    print(f"\nModel Computational Requirements:")
    print(f"FLOPs: {flops:,} ({flops/1e9:.2f} GFLOPs)")

    return flops


def train_epoch(model, device, train_loader, optimizer, epoch, logical_index_order, scheduler=None, threshold=0.0001):

    model.train()
    train_loss = 0
    correct = 0
    total = 0

    std_devs = {layer_name: [] for layer_name in model.layer_names}
    batch_indices_list = []

    start_time = time.time()
    total_images_processed = 0
    batch_processing_times = []

    num_active_batches = len(logical_index_order)
    print(f"Using {num_active_batches} batches this epoch")

    for loader_batch_idx, (data, target) in enumerate(train_loader):
        logical_batch_idx = logical_index_order[loader_batch_idx]

        batch_start_time = time.time()
        batch_indices_list.append(logical_batch_idx)

        data, target = data.to(device), target.to(device)

        # Optional: Mixup augmentation
        if np.random.random() > 0.5:
            lam = np.random.beta(0.2, 0.2)
            rand_idx = torch.randperm(data.size(0)).to(device)
            mixed_data = lam * data + (1 - lam) * data[rand_idx]
            target_a, target_b = target, target[rand_idx]

            optimizer.zero_grad()
            output, activations = model(mixed_data, return_activations=True)
            loss = lam * F.cross_entropy(output, target_a) + (1 - lam) * F.cross_entropy(output, target_b)
        else:
            optimizer.zero_grad()
            output, activations = model(data, return_activations=True)
            loss = F.cross_entropy(output, target)

        for layer_idx, act in enumerate(activations):
            std_dev = torch.std(act).item()
            std_devs[model.layer_names[layer_idx]].append(std_dev)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

        batch_end_time = time.time()
        batch_time = batch_end_time - batch_start_time
        batch_processing_times.append(batch_time)
        total_images_processed += data.size(0)

        if (loader_batch_idx + 1) % 100 == 0:
            print(f'Train Epoch: {epoch} [{total_images_processed}/{num_active_batches * data.size(0)} '
                  f'({100. * (loader_batch_idx + 1) / num_active_batches:.0f}%)]\t'
                  f'Loss: {loss.item():.6f}\tAccuracy: {100. * correct / total:.2f}%')

    end_time = time.time()
    training_time = end_time - start_time

    std_df = pd.DataFrame(std_devs)
    std_df['batch_idx'] = batch_indices_list
    std_df = std_df.set_index('batch_idx')
    std_df['mean_std'] = std_df.mean(axis=1)

    avg_batch_time = np.mean(batch_processing_times) if batch_processing_times else 0
    images_per_second = total_images_processed / training_time if training_time > 0 else 0

    epoch_loss = train_loss / len(batch_indices_list) if batch_indices_list else 0
    epoch_acc = 100. * correct / total if total > 0 else 0

    if scheduler is not None:
        scheduler.step()

    print(f"\nEpoch {epoch} completed in {training_time:.2f} seconds")
    print(f"Training Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")
    print(f"Throughput: {images_per_second:.2f} images/second, Avg batch time: {avg_batch_time:.4f} s")
    print(f"Current threshold: {threshold:.8f}")
    print(f"Batch composition: CONSISTENT across epochs (same samples per batch)")
    print(f"Intra-batch order: SHUFFLED to prevent memorization")

    throughput_metrics = {
        'epoch_time': training_time,
        'images_per_second': images_per_second,
        'avg_batch_time': avg_batch_time,
        'total_batches': len(batch_indices_list)
    }

    return epoch_loss, epoch_acc, std_df, batch_indices_list, throughput_metrics


def validate(model, device, data_loader, dataset_name="Validation"):
    model.eval()
    val_loss = 0
    correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)

            output = model(data)

            val_loss += F.cross_entropy(output, target).item()

            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()

            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    val_loss /= len(data_loader)
    val_acc = 100. * correct / len(data_loader.dataset)

    print(f"{dataset_name} Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")

    return val_loss, val_acc, all_targets, all_preds

def plot_training_metrics(epoch_times, train_losses, train_accs, val_losses, val_accs):
    plt.figure(figsize=(18, 12))

    plt.subplot(2, 2, 1)
    plt.plot(range(1, len(epoch_times) + 1), epoch_times, marker='o')
    plt.title('Training Time per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Time (seconds)')
    plt.grid(True)

    plt.subplot(2, 2, 3)
    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(val_accs, label='Validation Accuracy')
    plt.title('Model Accuracy (SVHN)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 4)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Model Loss (SVHN)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('training_metrics_svhn.pdf')
    plt.show()

def plot_confusion_matrix(all_targets, all_preds, class_names, title='Confusion Matrix'):
    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title(title)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_svhn.pdf')
    plt.show()

def calculate_exponential_threshold(delta_start, delta_end, T, t):
    """
    Calculate threshold using exponential function: δ(t) = δ_initial * e^(alpha * t)
    """
    alpha = math.log(delta_end / delta_start) / T
    threshold = delta_start * math.exp(alpha * t)
    return threshold


def plot_batch_prune_analysis(threshold_history, remaining_batches_history,
                            dropped_per_epoch_history, pruned_percentage_history):
    plt.figure(figsize=(20, 12))

    plt.subplot(3, 2, 1)
    plt.plot(range(1, len(threshold_history) + 1), threshold_history, marker='o', color='red')
    plt.title('Exponential Threshold Evolution Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Threshold Value')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 2, 2)
    plt.plot(range(1, len(remaining_batches_history) + 1), remaining_batches_history, marker='o', color='blue')
    plt.title('Remaining Batches per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Batches')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 2, 3)
    plt.bar(range(1, len(dropped_per_epoch_history) + 1), dropped_per_epoch_history, color='orange', alpha=0.7)
    plt.title('Batches Dropped per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Batches Dropped')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 2, 4)
    plt.plot(range(1, len(pruned_percentage_history) + 1), pruned_percentage_history,
             marker='d', color='green', linewidth=2)
    plt.title('Cumulative Pruned Percentage Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Pruned Percentage (%)')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 2, 5)
    plt.scatter(threshold_history, remaining_batches_history, alpha=0.6, color='coral')
    plt.title('Threshold vs Remaining Batches')
    plt.xlabel('Threshold Value')
    plt.ylabel('Remaining Batches')
    plt.xscale('log')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 2, 6)
    epochs = range(1, len(threshold_history) + 1)
    plt.semilogy(epochs, threshold_history, marker='s', color='purple', linewidth=2)
    plt.title('Exponential Threshold Growth Pattern')
    plt.xlabel('Epoch')
    plt.ylabel('Threshold Value (log scale)')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('batch_prune_exponential_threshold_analysis_svhn.pdf', dpi=300, bbox_inches='tight')
    plt.show()

def run_svhn_classification():
    # Configure GPU
    device = configure_gpu()

    # Hyperparameters
    total_epochs = 200
    batch_size = 128
    learning_rate = 0.05
    weight_decay = 5e-4
    momentum = 0.9

    # Exponential threshold parameters
    delta_start = 0.000001   # starting δ
    delta_end = 0.00005      # ending δ

    min_batch_percentage = 0

    # Load SVHN data
    full_train_dataset, val_loader, test_loader, class_names = load_images(batch_size, val_size=5000)

    global GLOBAL_TRAIN_BATCH_INDICES
    total_original_batches = len(GLOBAL_TRAIN_BATCH_INDICES)
    max_epochs = total_epochs

    # Compute actual train size for summary
    train_size = sum(len(b) for b in GLOBAL_TRAIN_BATCH_INDICES)

    print(f"Total original training batches per epoch: {total_original_batches}")
    print(f"Maximum number of epochs: {max_epochs}")

    # Create ResNet18 model with 10 output classes
    model = ResNet18(num_classes=10).to(device)

    print(model)

    total_params, trainable_params = calculate_model_parameters(model)

    try:
        flops = calculate_flops(model)
    except ImportError:
        print("thop package not installed. Skipping FLOPs calculation.")
        flops = None

    optimizer = optim.SGD(model.parameters(),
                         lr=learning_rate,
                         momentum=momentum,
                         weight_decay=weight_decay,
                         nesterov=True)

    scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=1e-5)

    # Training and validation history
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []

    epoch_times = []
    batch_counts = []

    # Initialize tracking for batch dropping
    prev_epoch_std_df = None
    batch_indices_to_use = list(range(total_original_batches))
    total_dropped_batches = 0
    total_remaining_batches = []
    total_batches_dropped_per_epoch = []

    threshold_history = []
    pruned_percentage_history = []

    batch_std_history = {}

    total_training_start = time.time()

    best_val_acc = 0
    best_model_state = None

    for epoch in range(1, total_epochs + 1):
        print(f"\n{'='*50}")
        print(f"EPOCH {epoch}/{total_epochs}")
        print(f"{'='*50}")

        current_threshold = calculate_exponential_threshold(delta_start, delta_end, total_epochs, epoch)

        threshold_history.append(current_threshold)

        print(f"Current threshold (δ({epoch})): {current_threshold:.8f}")

        train_loader, logical_index_order = build_train_loader_for_epoch(
            full_train_dataset, batch_indices_to_use, batch_size, shuffle_within_batch=True
        )

        # Train for one epoch
        train_loss, train_acc, current_std_df, used_batch_indices, throughput_metrics = train_epoch(
            model, device, train_loader, optimizer, epoch,
            logical_index_order=logical_index_order,
            scheduler=scheduler,
            threshold=current_threshold
        )

        epoch_times.append(throughput_metrics['epoch_time'])
        batch_counts.append(throughput_metrics['total_batches'])
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        current_remaining_batches = len(used_batch_indices) if used_batch_indices is not None else total_original_batches
        total_remaining_batches.append(current_remaining_batches)

        current_pruned_percentage = (1 - current_remaining_batches / total_original_batches) * 100
        pruned_percentage_history.append(current_pruned_percentage)

        # Evaluate on VALIDATION set
        val_loss, val_acc, _, _ = validate(model, device, val_loader, dataset_name="Validation")
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {key: value.cpu().clone() for key, value in model.state_dict().items()}
            print(f"New best validation accuracy: {best_val_acc:.2f}%")
            torch.save(model.state_dict(), 'best_resnet18_svhn_exponential_threshold.pth')

        batches_dropped_this_epoch = 0

        # After the first epoch, start the batch dropping mechanism
        if epoch > 1 and prev_epoch_std_df is not None:
            common_indices = set(prev_epoch_std_df.index).intersection(set(current_std_df.index))

            batches_to_drop = []

            for idx in common_indices:
                prev_mean_std = prev_epoch_std_df.loc[idx, 'mean_std']
                curr_mean_std = current_std_df.loc[idx, 'mean_std']

                if idx not in batch_std_history:
                    batch_std_history[idx] = []
                batch_std_history[idx].append(curr_mean_std)

                if abs(prev_mean_std - curr_mean_std) <= current_threshold:
                    batches_to_drop.append(idx)

            if batches_to_drop:
                print(f"\nDropping {len(batches_to_drop)} batches for the next epoch:")
                print(f"Batch indices: {batches_to_drop[:10]}{'...' if len(batches_to_drop) > 10 else ''}")
                total_dropped_batches += len(batches_to_drop)
                batches_dropped_this_epoch = len(batches_to_drop)
            else:
                print("\nNo batches to drop for the next epoch")

            batch_indices_to_use = [i for i in batch_indices_to_use if i not in set(batches_to_drop)]

        prev_epoch_std_df = current_std_df

        total_batches_dropped_per_epoch.append(batches_dropped_this_epoch)

        percent_remaining = len(batch_indices_to_use) / total_original_batches * 100
        print(f"\nBatches remaining: {len(batch_indices_to_use)}/{total_original_batches} ({percent_remaining:.2f}%)")
        print(f"Total batches dropped so far: {total_dropped_batches}/{total_original_batches} ({total_dropped_batches/total_original_batches*100:.2f}%)")

        if epoch % 10 == 0:
            plot_batch_std_distribution(current_std_df, batch_indices_to_use, epoch)

        # Early stopping if we've dropped too many batches
        if len(batch_indices_to_use) <= total_original_batches * min_batch_percentage:
            print(f"\nStopping early as fewer than {min_batch_percentage*100}% of original batches remain")
            break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model with validation accuracy: {best_val_acc:.2f}%")

    # Calculate total training time
    total_training_end = time.time()
    total_training_time = total_training_end - total_training_start
    average_epoch_time = sum(epoch_times) / len(epoch_times) if epoch_times else 0

    # Calculate Data Utilization Index (DUI)
    total_epochs_completed = len(epoch_times)
    sum_remaining_batches = sum(total_remaining_batches)
    max_possible_batches = max_epochs * total_original_batches
    data_utilization_index = sum_remaining_batches / max_possible_batches
    data_savings_index = 1 - data_utilization_index

    # Plot batch pruning analysis
    plot_batch_prune_analysis(threshold_history, total_remaining_batches,
                            total_batches_dropped_per_epoch, pruned_percentage_history)

    plot_batch_dropping_analysis(total_remaining_batches, total_batches_dropped_per_epoch, epoch_times, batch_std_history)

    # Final stats
    percent_dropped = total_dropped_batches / total_original_batches * 100
    percent_remaining = 100 - percent_dropped
    total_remaining_batch = total_original_batches - total_dropped_batches
    print(f"\nData Utilization Index (DUI): {data_utilization_index:.4f}")
    print(f"\nData Savings Index (DSI): {data_savings_index:.4f}")
    print(f"Sum of remaining batches across all epochs: {sum_remaining_batches}")
    print(f"Maximum possible batches (max_epochs * total_batches): {max_possible_batches}")

    print(f"\nFinal Statistics:")
    print(f"Total batches dropped: {total_dropped_batches}/{total_original_batches} ({percent_dropped:.2f}%)")
    print(f"Total batches remaining: {total_remaining_batch}/{total_original_batches} ({percent_remaining:.2f}%)")
    print(f"Final threshold: {threshold_history[-1]:.8f}")

    print(f"\nTraining Time Statistics:")
    print(f"Total training time: {total_training_time:.2f} seconds ({total_training_time/60:.2f} minutes)")
    print(f"Average epoch time: {average_epoch_time:.2f} seconds")
    print(f"Number of epochs completed: {len(epoch_times)}")

    print(f"\nModel Statistics:")
    print(f"Total parameters: {total_params:,}")
    if flops:
        print(f"FLOPs: {flops:,} ({flops/1e9:.2f} GFLOPs)")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    # Final validation
    print(f"\n{'='*50}")
    print("FINAL VALIDATION SET EVALUATION")
    print(f"{'='*50}")
    final_val_loss, final_val_acc, final_val_targets, final_val_preds = validate(
        model, device, val_loader, dataset_name="Final Validation"
    )

    # Final test
    print(f"\n{'='*50}")
    print("FINAL TEST SET EVALUATION")
    print(f"{'='*50}")
    final_test_loss, final_test_acc, final_test_targets, final_test_preds = validate(
        model, device, test_loader, dataset_name="Final Test"
    )

    print(f"\nFinal Results Summary:")
    print(f"  Validation Loss: {final_val_loss:.4f}, Accuracy: {final_val_acc:.2f}%")
    print(f"  Test Loss: {final_test_loss:.4f}, Accuracy: {final_test_acc:.2f}%")

    # Plot confusion matrix for TEST set
    plot_confusion_matrix(final_test_targets, final_test_preds, class_names, title='SVHN Test Set Confusion Matrix')

    # Plot training metrics
    plot_training_metrics(epoch_times, train_losses, train_accs, val_losses, val_accs)

    # Save summary
    summary_data = {
        'dataset': ['SVHN'],
        'num_classes': [10],
        'total_training_time': [total_training_time],
        'average_epoch_time': [average_epoch_time],
        'epochs_completed': [total_epochs_completed],
        'best_validation_accuracy': [best_val_acc],
        'final_validation_accuracy': [final_val_acc],
        'final_test_accuracy': [final_test_acc],
        'total_parameters': [total_params],
        'data_utilization_index': [data_utilization_index],
        'data_savings_index': [data_savings_index],
        'total_batches_dropped': [total_dropped_batches],
        'total_remaining_batches': [total_remaining_batch],
        'batch_drop_percentage': [percent_dropped],
        'final_threshold': [threshold_history[-1] if threshold_history else delta_start],
        'learning_rate': [learning_rate],
        'weight_decay': [weight_decay],
        'train_samples': [train_size],
        'val_samples': [5000],
        'test_samples': [len(test_loader.dataset)]
    }

    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv('exponential_threshold_batch_dropping_summary_svhn.csv', index=False)
    print("\nSummary saved to exponential_threshold_batch_dropping_summary_svhn.csv")

    # Save threshold history
    threshold_df = pd.DataFrame({
        'epoch': range(1, len(threshold_history) + 1),
        'threshold': threshold_history,
        'remaining_batches': total_remaining_batches,
        'batches_dropped': total_batches_dropped_per_epoch,
        'pruned_percentage': pruned_percentage_history
    })
    threshold_df.to_csv('exponential_threshold_history_svhn.csv', index=False)
    print("Threshold history saved to exponential_threshold_history_svhn.csv")

    return model, summary_df

def plot_batch_std_distribution(std_df, remaining_batch_indices, epoch):
    plt.figure(figsize=(10, 6))

    remaining_stds = std_df.loc[std_df.index.isin(remaining_batch_indices), 'mean_std']
    dropped_stds = std_df.loc[~std_df.index.isin(remaining_batch_indices), 'mean_std']

    plt.hist(remaining_stds, bins=30, alpha=0.5, label='Remaining Batches')
    if len(dropped_stds) > 0:
        plt.hist(dropped_stds, bins=30, alpha=0.5, label='Dropped Batches')

    plt.title(f'Batch Std Distribution - Epoch {epoch} (SVHN)')
    plt.xlabel('Mean Standard Deviation')
    plt.ylabel('Number of Batches')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f'batch_std_distribution_epoch_{epoch}_svhn.pdf')
    plt.close()

def plot_batch_dropping_analysis(remaining_batches, dropped_per_epoch, epoch_times, batch_std_history):
    plt.figure(figsize=(18, 12))

    plt.subplot(2, 2, 1)
    plt.plot(range(1, len(remaining_batches) + 1), remaining_batches, marker='o')
    plt.title('Remaining Batches per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Batches')
    plt.grid(True)

    plt.subplot(2, 2, 2)
    plt.bar(range(1, len(dropped_per_epoch) + 1), dropped_per_epoch)
    plt.title('Batches Dropped per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Batches Dropped')
    plt.grid(True)

    plt.subplot(2, 2, 3)
    plt.scatter(remaining_batches, epoch_times)
    plt.title('Training Time vs Remaining Batches')
    plt.xlabel('Number of Batches')
    plt.ylabel('Epoch Time (seconds)')
    plt.grid(True)

    plt.subplot(2, 2, 4)
    sample_batches = list(batch_std_history.keys())[:10]
    for batch_idx in sample_batches:
        std_values = batch_std_history[batch_idx]
        plt.plot(range(1, len(std_values) + 1), std_values, label=f'Batch {batch_idx}')

    plt.title('Std Deviation Trend for Sample Batches')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Std Deviation')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('batch_dropping_analysis_svhn.pdf')
    plt.close()

if __name__ == "__main__":
    torch.manual_seed(42)
    np.random.seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    model, summary_df = run_svhn_classification()